In [5]:
%load_ext autoreload
%autoreload 2

from utils import * 
from models import *
from metrics import *

In [6]:
utils.set_seed()
graph_path="./data/dataset/final_graph_100k.pkl"

g, node_dict, edge_dict, labels, num_classes, train_idx, val_idx, \
test_idx, train_mask, val_mask, test_mask, target = utils.load_graph(graph_path)

ratings_path = "./data/dataset/ratings_df_100k.csv"
ratings, train_ratings, val_ratings, test_ratings  = \
    utils.load_rating(ratings_path, train_idx, val_idx, test_idx)


[INFO] Global seed set to 42


/home/amin/miniconda3/envs/gnn/lib/python3.8/site-packages/torch/storage.py:414: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(io.BytesIO(b))


In [7]:

input_dim = 256
features = {}
G, input_dims = init_feat(g, input_dim, features)

print(G.nodes['actor'].data['x'].size(), input_dims['actor'])
print(G.nodes['movie'].data['x'].size(), input_dims['movie'])
print(G.nodes['director'].data['x'].size(), input_dims['director'])
print(G.nodes['user'].data['x'].size(), input_dims['user'])


device = torch.device(f"cuda:0" if torch.cuda.is_available() else 'cpu')
print(device)

G = G.to(device)
labels = labels.to(device)

input_dims = {ntype: G.nodes[ntype].data['x'].shape[1] for ntype in G.ntypes}
input_dims

torch.Size([1728, 256]) 256
torch.Size([1661, 256]) 256
torch.Size([1077, 256]) 256
torch.Size([943, 256]) 256
cuda:0


{'actor': 256, 'director': 256, 'movie': 256, 'user': 256}

In [9]:
train_g = build_train_graph(g, train_idx)

train_user_pos = build_user_pos_dict(
    train_g,
    user_ntype='user',
    movie_ntype='movie',
    rating_etypes=('rate_4', 'rate_5')
)


In [10]:
test_user_pos = build_user_pos_dict(
    g,  # از خود گراف اصلی
    user_ntype='user',
    movie_ntype='movie',
    rating_etypes=('rate_4', 'rate_5')
)


In [11]:

hidden_dim = 256
num_classes = 3
results = []

for i in range(10):
    print('#' *10, i+1, '#' *10)
    # 1. load pretrained models
    path_model_gcn = 'best_gcn_model.pth'
    path_model_hgt = 'best_hgt_model.pth'


    model_a_trained = GCN_MTL(G, node_dict, input_dims, hidden_dim, num_classes)
    model_b_trained = HGT_MTL(G, node_dict, input_dims, hidden_dim, num_classes)

    model_a_trained.load_state_dict(torch.load(path_model_gcn))
    model_b_trained.load_state_dict(torch.load(path_model_hgt))

    final_model = DualBranchCrossAttnModel(
        model_a_trained,  # GCN
        model_b_trained,  # HGT
        hidden_dim=256,
        num_classes=3,
        alpha_cls=0.95,   #  HGT
        beta_rec=0.95     #  GCN
    ).to(device)

    # freeze backbones
    for p in final_model.branch_gcn.parameters():
        p.requires_grad = False
    for p in final_model.branch_hgt.parameters():
        p.requires_grad = False

    train(
        final_model,
        G,
        train_idx,
        val_idx,
        val_ratings,
        labels,
        train_user_pos,
        train_ratings,
        epochs=100,
        lr=1e-3,
        device=device
    )
    result = test(
        final_model,
        G,
        test_idx,
        labels,
        test_ratings)

    results.append(result)

########## 1 ##########


/tmp/ipykernel_11897/1668897131.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_a_trained.load_state_dict(torch.load(path_model_gcn))
/tmp/ipykernel_11897/1668897

Epoch 010 | CLS: 0.183 | MSE: 0.064 | VAL: 0.2119
Epoch 020 | CLS: 0.123 | MSE: 0.067 | VAL: 0.1636
Epoch 030 | CLS: 0.117 | MSE: 0.053 | VAL: 0.1510
Epoch 040 | CLS: 0.115 | MSE: 0.063 | VAL: 0.1445
Epoch 050 | CLS: 0.113 | MSE: 0.060 | VAL: 0.1413
Epoch 060 | CLS: 0.102 | MSE: 0.053 | VAL: 0.1386
Epoch 070 | CLS: 0.085 | MSE: 0.054 | VAL: 0.1375
Epoch 080 | CLS: 0.088 | MSE: 0.055 | VAL: 0.1367
Epoch 090 | CLS: 0.101 | MSE: 0.059 | VAL: 0.1356
Early stopping!


/home/amin/miniconda3/envs/gnn/lib/python3.8/site-packages/sklearn/cluster/_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


########## 2 ##########


...

/home/amin/miniconda3/envs/gnn/lib/python3.8/site-packages/sklearn/cluster/_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


########## 10 ##########


/tmp/ipykernel_11897/1668897131.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_a_trained.load_state_dict(torch.load(path_model_gcn))
/tmp/ipykernel_11897/1668897

Epoch 010 | CLS: 0.192 | MSE: 0.067 | VAL: 0.2119
Epoch 020 | CLS: 0.131 | MSE: 0.059 | VAL: 0.1617
Epoch 030 | CLS: 0.119 | MSE: 0.059 | VAL: 0.1491
Epoch 040 | CLS: 0.114 | MSE: 0.051 | VAL: 0.1441
Epoch 050 | CLS: 0.105 | MSE: 0.050 | VAL: 0.1390
Epoch 060 | CLS: 0.098 | MSE: 0.054 | VAL: 0.1356
Epoch 070 | CLS: 0.091 | MSE: 0.054 | VAL: 0.1384
Early stopping!


/home/amin/miniconda3/envs/gnn/lib/python3.8/site-packages/sklearn/cluster/_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


In [12]:
df_MLT = pd.DataFrame(results)
df_MLT_Mean10 = df_MLT.drop(columns='Name').mean()
df_MLT_Mean10

Accuracy    0.974950
Micro-F1    0.974950
Macro-F1    0.974356
NMI         0.848530
ARI         0.893000
RMSE        0.922960
MAE         0.733105
dtype: float64

In [6]:
import torch
import dgl
import numpy
import pandas
import sklearn
import pickle
print(torch.__version__)
print(dgl.__version__)
print(numpy.__version__)
print(pandas.__version__)
print(sklearn.__version__)
print(pickle.HIGHEST_PROTOCOL)

2.4.0+cu121
2.4.0+cu118
1.24.4
2.0.3
1.3.2
5
